In [5]:
import json

def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [6]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

In [8]:
from dataclasses import dataclass

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int  # epoch milliseconds

In [9]:
test_bytes = json.dumps({
    'PULocationID': 186,
    'DOLocationID': 79,
    'trip_distance': 1.72,
    'total_amount': 17.31,
    'tpep_pickup_datetime': 1730429702000
}).encode('utf-8')

ride_deserializer(test_bytes)
# Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72,
#      total_amount=17.31, tpep_pickup_datetime=1730429702000)

Ride(PULocationID=186, DOLocationID=79, trip_distance=1.72, total_amount=17.31, tpep_pickup_datetime=1730429702000)

In [10]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

Listening to rides...
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 19:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 19:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 19:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-10-31 19:13:25
Received: PU=142, DO=237, distance=2.28, amount=$24.94, pickup=2025-10-31 19:49:07
Received: PU=163, DO=238, distance=2.7, amount=$25.62, pickup=2025-10-31 19:07:19
Received: PU=138, DO=261, distance=12.87, amount=$86.14, pickup=2025-10-31 19:00:00
Received: PU=138, DO=37, distance=8.4, amount=$48.65, pickup=2025-10-31 19:18:50
Received: PU=90, DO=100, distance=0.85, amount=$16.45, pickup=2025-10-31 19:21:11
Received: PU=142, DO=170, distance=3.01, amount=$25.85, pickup=2025-10-31 19:07:31

... received 10 messages so far (stopping after 10 for demo)
